# Семинар 3. NumPy: случайные выборки, статистики, пропуски, поиск и сортировка

- Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-03.ipynb`, например `2MP9-Ivanov-Ivan-seminar-03.ipynb`.
- Порядок выполнения важен: картотека на 1000 дел создаётся в разделе 2, пропуски добавляются в разделе 4, и всё это используется до конца ноутбука.
- Упражнения к занятию лежат в папке `exercises/`: листок `seminar-03-tasks.md` с формулировками и ноутбук `seminar-03-tasks.ipynb` с заготовками и тестами. Они начинаются на паре сразу после показа. Ответы вводятся в тест Moodle «Семинар 3».
- Шпаргалка по функциям: `reference/numpy.md`.

## 1. Случайные числа и выборки

На семинарах 1 и 2 генератор с зерном давал всем одни и те же «случайные» данные. Сегодня он нужен сам по себе: юристу приходится выбирать дела случайно, когда проверить все нельзя, и распределять дела между судьями так, чтобы никто не мог выбрать себе удобное. Генератор создаётся один раз функцией `default_rng`. **Зерно** делает результат воспроизводимым: с тем же зерном те же числа, без зерна каждый раз новые.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
rng.integers(10_000, 1_000_000, size=5)   # пять случайных цен исков

Выборочная проверка: из двенадцати дел взять три случайно и без повторов. Метод `choice` выбирает элементы из массива; `replace=False` запрещает повторы, то есть выбор идёт без возвращения. Без этого параметра выбор идёт с возвращением, и во второй ячейке одно дело попадает в выборку дважды.

In [ ]:
numbers = np.arange(1, 13)                     # номера дел

rng.choice(numbers, size=3, replace=False)     # три дела на проверку

In [ ]:
rng.choice(numbers, size=4)                    # с возвращением: повторы возможны

Жеребьёвка: автоматизированное распределение дел между судьями начинается с перемешивания. `permutation` возвращает перемешанную копию массива, исходный порядок не меняется. Первые четыре дела в новом порядке достаются первому судье, следующие четыре — второму, остальные — третьему.

In [ ]:
order = rng.permutation(numbers)               # дела в случайном порядке
order

In [ ]:
order[:4], order[4:8], order[8:]               # дела трёх судей

Воспроизводимость: новый генератор с тем же зерном 3 повторяет всё с начала, и первые пять цен совпадают с первой ячейкой раздела. Генератор без зерна даёт при каждом запуске новые числа: выполните последнюю ячейку дважды.

In [ ]:
np.random.default_rng(3).integers(10_000, 1_000_000, size=5)   # те же пять цен

In [ ]:
np.random.default_rng().integers(10_000, 1_000_000, size=5)    # каждый раз новые

## 2. Картотека на 1000 дел

Картотека судебного участка за год: тысяча дел. Цены исков собраны из двух частей: 950 обычных дел до 300 000 рублей и 50 крупных от одного до пяти миллионов; `np.append` соединяет два массива, `shuffle` перемешивает результат на месте и ничего не возвращает: запись `amounts = rng.shuffle(amounts)` уничтожила бы картотеку, в `amounts` оказалось бы `None`. Сроки и исходы устроены как на семинаре 2, у каждого дела есть номер. Такой массив уже не прочитать глазами: NumPy печатает начало и конец и ставит многоточие.

In [ ]:
rng = np.random.default_rng(2026)

amounts = np.append(rng.integers(10_000, 300_000, size=950),
                    rng.integers(1_000_000, 5_000_000, size=50))   # цены исков, руб.
rng.shuffle(amounts)                                                # перемешать на месте
days = rng.integers(10, 300, size=1000)                             # срок рассмотрения, дней
outcomes = rng.integers(0, 2, size=1000)                            # 1 удовлетворён, 0 отказано
numbers = np.arange(1, 1001)                                        # номера дел

amounts

In [ ]:
amounts.size, amounts.min(), amounts.max()

Минимум и максимум ещё видны, а «сколько стоит типичное дело» и «сколько оно обычно длится» из тысячи чисел не считываются. Нужны сводные числа — **статистики**.

## 3. Описательные статистики

**Среднее** — сумма цен, делённая на число дел. **Медиана** — цена дела посередине упорядоченного списка: половина дел дешевле, половина дороже. На картотеке они расходятся почти вдвое: пятьдесят крупных исков тянут среднее вверх, а медиана остаётся ценой типичного дела. В отчёте юрист должен понимать, какое из двух чисел он называет.

In [ ]:
amounts.mean(), np.median(amounts)   # среднее и медиана

У массива нет метода `median`: медиана — функция `np.median`. Ошибка ниже оставлена намеренно, её получит каждый, кто напишет по аналогии с `mean`.

In [ ]:
amounts.median()

**Стандартное отклонение** `std` — типичное отклонение от среднего: чем оно больше, тем сильнее разброс цен. **Процентиль**: 90-й процентиль сроков — срок, быстрее которого рассмотрены 90 % дел; так формулируют показатели судебной статистики. Квартили — 25-й, 50-й и 75-й процентили, `percentile` принимает и список; 50-й процентиль и есть медиана.

In [ ]:
amounts.std()   # разброс цен

In [ ]:
np.percentile(days, 90)   # срок 90 % дел

In [ ]:
np.percentile(amounts, [25, 50, 75])   # квартили цен

Доля удовлетворённых исков и средний срок считаются как на семинаре 2: среднее по массиву из нулей и единиц и среднее по срокам.

In [ ]:
outcomes.mean(), days.mean()   # доля удовлетворённых, средний срок

## 4. Пропуски

В настоящей картотеке часть дел не завершена: срок и исход ещё не известны. Для «неизвестно» в NumPy есть значение `np.nan`, not a number. Оно дробное, поэтому массив целых чисел перед записью `nan` переводится в дробный тип через `astype(float)`. Ниже пятьдесят случайных дел объявляются незавершёнными: `choice` по числу 1000 выбирает индексы.

In [ ]:
unfinished = rng.choice(1000, size=50, replace=False)   # индексы незавершённых дел

days = days.astype(float)
outcomes = outcomes.astype(float)
days[unfinished] = np.nan
outcomes[unfinished] = np.nan

print(days[:10])        # начало картотеки
days[unfinished[:3]]    # сроки трёх незавершённых дел

Пропуски находит только `np.isnan`: сравнение с `np.nan` не работает, потому что `nan` не равен ничему, даже самому себе. Сумма маски пропусков — их число, среднее — их доля.

In [ ]:
np.nan == np.nan   # nan не равен даже себе

In [ ]:
np.isnan(days).sum(), np.isnan(days).mean()   # число и доля незавершённых

Обычные статистики при пропусках молча возвращают `nan`: ошибки нет, а результата тоже нет. Два выхода: функции с приставкой `nan`, которые пропуски пропускают, или отбор известных значений маской.

In [ ]:
days.mean()   # nan: пропуски ломают среднее

In [ ]:
np.nanmean(days), np.nanmedian(days), np.nanpercentile(days, 90)   # без пропусков

In [ ]:
days[~np.isnan(days)].mean()   # то же через маску

Иногда пропуски заполняют: незавершённым делам временно приписывают типичный срок, медиану известных. Работаем с копией, картотека остаётся честной; к заполнению пропусков вернёмся на семинаре 10.

In [ ]:
filled = days.copy()
filled[np.isnan(filled)] = np.nanmedian(days)

np.isnan(filled).sum(), filled.mean()   # пропусков нет, среднее считается

## 5. Поиск

`np.where(условие, а, б)` собирает новый массив: там, где условие истинно, берётся `а`, иначе `б`. Так число превращается в категорию: дела до 100 000 рублей включительно идут в упрощённом порядке по ст. 232.2 ГПК РФ, остальные — в общем. Ниже метки дел с одиннадцатого по двадцатое: среди них есть и те, и другие.

In [ ]:
procedure = np.where(amounts <= 100_000, 'упрощённое', 'общее')   # порядок производства
procedure[10:20]

In [ ]:
(procedure == 'упрощённое').sum()   # дел в упрощённом порядке

`argmax` и `argmin` возвращают не значение, а **индекс** наибольшего и наименьшего элемента. По индексу из массива номеров достаётся номер дела: так находят самое дорогое дело, а не только его цену.

In [ ]:
np.argmax(amounts), amounts.max()   # индекс и значение

In [ ]:
numbers[np.argmax(amounts)]   # номер самого дорогого дела

При пропусках `argmax` и `argmin` возвращают индекс первого `nan`: так устроены сами функции, они считают пропуск и наибольшим, и наименьшим. Ошибки нет, ответ выглядит правдоподобно и при этом неверен. Для массивов с пропусками нужны `nanargmax` и `nanargmin`.

In [ ]:
days[np.argmax(days)]   # argmax указывает на nan

In [ ]:
numbers[np.nanargmax(days)], np.nanmax(days)   # самое долгое завершённое дело

## 6. Сортировка

`np.sort` возвращает отсортированную копию по возрастанию; срез `[::-1]` разворачивает порядок. `argsort` возвращает не значения, а индексы в порядке возрастания: через них один массив сортируется по другому, и номера дел выстраиваются по цене иска. В pandas та же операция называется `sort_values`.

In [ ]:
np.sort(amounts)[:5]   # пять самых дешёвых цен

In [ ]:
np.sort(amounts)[::-1][:5]   # пять самых дорогих цен

In [ ]:
order = np.argsort(amounts)[::-1]   # индексы по убыванию цены
numbers[order[:5]]                  # номера пяти самых дорогих дел

In [ ]:
days[order[:5]]   # сроки этих пяти дел

`np.sort` ставит `nan` в конец, поэтому самые долгие дела ищут через `nanargmax` или отбирают маской, а не хвостом сортировки.

In [ ]:
np.sort(days)[-3:]   # nan в конце

## 7. Мини-анализ картотеки

Всё вместе, одной страницей: что можно сказать о тысяче дел, не читая их. Паспорт картотеки собирает статистики разделов 3 и 4; `nanstd` — стандартное отклонение без пропусков.

In [ ]:
print('Дел в картотеке:', numbers.size)
print('Не завершено:', int(np.isnan(days).sum()))
print('Типичная цена иска, руб.:', np.median(amounts))
print('Средняя цена иска, руб.:', round(amounts.mean()))
print('Четверть дел дешевле, руб.:', np.percentile(amounts, 25))
print('Типичный срок, дней:', np.nanmedian(days))
print('Разброс сроков, дней:', round(np.nanstd(days)))
print('90 % дел быстрее, дней:', np.nanpercentile(days, 90))
print('Удовлетворено исков, доля:', round(np.nanmean(outcomes), 2))

Два вопроса, на которые картотека отвечает одной строкой каждый. Незавершённые дела дороже завершённых? Маска пропусков отбирает цены тех и других, медианы сравниваются. Среди самых дорогих дел незавершённых больше, чем в среднем по картотеке? Индексы пятидесяти самых дорогих дел через `argsort`, по ним сроки, доля пропусков среди них против 0.05 по всей картотеке.

In [ ]:
unfinished_mask = np.isnan(days)
np.median(amounts[unfinished_mask]), np.median(amounts[~unfinished_mask])   # цена незавершённых и завершённых

In [ ]:
top50 = np.argsort(amounts)[::-1][:50]   # индексы 50 самых дорогих дел
np.isnan(days[top50]).mean()             # доля незавершённых среди них

In [ ]:
long = days > np.nanpercentile(days, 95)   # дела дольше 95 % остальных
numbers[long].size, numbers[long][:10]     # сколько их и первые десять

Дела дольше 95-го процентиля — кандидаты в выбросы; на семинаре 10 к ним вернёмся с более строгими критериями. Ответы на два вопроса выше получены по случайным данным: на настоящей картотеке они и были бы результатом анализа.

## Итоги

Генератор с зерном даёт воспроизводимые выборки и жеребьёвку; медиана и процентили описывают картотеку честнее среднего; пропуски ищут только через `isnan`, а считают функциями с приставкой `nan`; `where` размечает, `argmax` и `argsort` находят и упорядочивают дела по номерам.